In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Read Bronze table

bronze_df = spark.table("bronze.weather_raw")

display(bronze_df)

In [0]:
json_schema = StructType([
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),

    StructField("hourly", StructType([

        StructField("time", ArrayType(StringType())),

        StructField("temperature_2m", ArrayType(DoubleType())),

        StructField("relative_humidity_2m", ArrayType(IntegerType())),

        StructField("precipitation", ArrayType(DoubleType())),

        StructField("wind_speed_10m", ArrayType(DoubleType())),

        StructField("cloud_cover", ArrayType(IntegerType()))

    ]))
])

In [0]:
parsed_df = bronze_df.withColumn(
    "weather_data",
    from_json(col("raw_json"), json_schema)
)

display(parsed_df)

In [0]:
exploded_df = parsed_df.select(

    col("location"),

    explode(

        arrays_zip(

            col("weather_data.hourly.time"),

            col("weather_data.hourly.temperature_2m"),

            col("weather_data.hourly.relative_humidity_2m"),

            col("weather_data.hourly.precipitation"),

            col("weather_data.hourly.wind_speed_10m"),

            col("weather_data.hourly.cloud_cover")

        )

    ).alias("weather")

)

display(exploded_df)

In [0]:
silver_df = exploded_df.select(

    col("location"),

    col("weather.time").alias("weather_time"),

    col("weather.temperature_2m").alias("temperature"),

    col("weather.relative_humidity_2m").alias("humidity"),

    col("weather.precipitation").alias("rainfall"),

    col("weather.wind_speed_10m").alias("wind_speed"),

    col("weather.cloud_cover").alias("cloud_cover")

)

display(silver_df)

In [0]:
silver_df = silver_df.withColumn(
    "weather_time",
    to_timestamp("weather_time")
)

# Remove records with missing critical fields

silver_df = silver_df.filter(
    col("weather_time").isNotNull() &
    col("temperature").isNotNull()
)

# Remove duplicate records

silver_df = silver_df.dropDuplicates(
    ["location", "weather_time"]
)

# Add processing timestamp

silver_df = silver_df.withColumn(
    "processed_at",
    current_timestamp()
)

display(silver_df)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.weather_cleaned")

print("Silver layer loaded successfully!")

In [0]:
%sql
SELECT COUNT(*)
FROM silver.weather_cleaned;